# Тренажер: розгортка призми з різнобічною основою

У цій версії основа призми — **довільний опуклий многокутник** із різними сторонами та кутами. Призма є прямою: бічні ребра перпендикулярні до площин основ.

Студент задає координати вершин основи в порядку обходу контуру та висоту `H`. Програма автоматично:
- обчислює довжини сторін і периметр основи;
- будує просторову модель;
- розгортає бічну поверхню в послідовність прямокутників;
- приєднує дві копії основи до відповідних ребер першої бічної грані.

**Геометричний принцип:** ширина кожної бічної грані дорівнює довжині відповідної сторони основи, а висота всіх бічних граней дорівнює `H`. Основа не зобов'язана бути правильною.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import Textarea, FloatSlider, Checkbox, Button, Output, VBox
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

DEFAULT_BASE = np.array([
    [-2.2, -1.2],
    [ 1.2, -1.6],
    [ 2.4,  0.4],
    [ 0.7,  2.0],
    [-1.8,  1.4]
], dtype=float)

def rotate_to_first_edge(points):
    """Повертає контур так, щоб перша сторона була горизонтальною."""
    e = points[1] - points[0]
    ang = np.arctan2(e[1], e[0])
    rot = np.array([[np.cos(-ang), -np.sin(-ang)],
                    [np.sin(-ang),  np.cos(-ang)]])
    return (points - points[0]) @ rot.T

def is_convex_polygon(points):
    crosses = []
    n = len(points)
    for i in range(n):
        v1 = points[(i+1) % n] - points[i]
        v2 = points[(i+2) % n] - points[(i+1) % n]
        crosses.append(np.cross(v1, v2))
    crosses = np.asarray(crosses)
    return np.all(crosses > 1e-8) or np.all(crosses < -1e-8)

def build_prism_net(points, H=4.0, show_labels=True):
    points = np.asarray(points, dtype=float)
    n = len(points)
    if n < 3:
        raise ValueError("Потрібно щонайменше три вершини.")
    if any(np.linalg.norm(points[(i+1) % n] - points[i]) < 1e-8 for i in range(n)):
        raise ValueError("Сусідні вершини не можуть збігатися.")
    if not is_convex_polygon(points):
        raise ValueError("Основа має бути опуклою, а вершини — введені послідовно по контуру.")

    edges = np.roll(points, -1, axis=0) - points
    lengths = np.linalg.norm(edges, axis=1)
    perimeter = lengths.sum()

    lower = np.column_stack([points, np.zeros(n)])
    upper = lower.copy()
    upper[:, 2] = H

    fig = plt.figure(figsize=(14, 6.5))
    ax3d = fig.add_subplot(121, projection="3d")
    axnet = fig.add_subplot(122)

    faces = []
    for i in range(n):
        j = (i+1) % n
        faces.append([lower[i], lower[j], upper[j], upper[i]])
        ax3d.plot([lower[i,0], lower[j,0], upper[j,0], upper[i,0], lower[i,0]],
                  [lower[i,1], lower[j,1], upper[j,1], upper[i,1], lower[i,1]],
                  [lower[i,2], lower[j,2], upper[j,2], upper[i,2], lower[i,2]],
                  color="steelblue", lw=1.4)
    faces.extend([[lower[i] for i in range(n)], [upper[i] for i in range(n)]])
    ax3d.add_collection3d(Poly3DCollection(faces, alpha=.12,
                                           facecolor="cornflowerblue", edgecolor="none"))
    lo = np.vstack([lower, lower[0]])
    up = np.vstack([upper, upper[0]])
    ax3d.plot(lo[:,0], lo[:,1], lo[:,2], color="steelblue", lw=1.8)
    ax3d.plot(up[:,0], up[:,1], up[:,2], color="steelblue", lw=1.8)
    ax3d.set_title("Просторова модель прямої призми")
    ax3d.set_xlabel("X"); ax3d.set_ylabel("Y"); ax3d.set_zlabel("Z")
    xyz = np.vstack([lower, upper])
    ranges = np.ptp(xyz, axis=0)
    ax3d.set_box_aspect(np.maximum(ranges, 0.5))
    ax3d.view_init(elev=22, azim=-55)

    # Бічна поверхня: прямокутники шириною в довжину відповідного ребра.
    cursor = 0.0
    for i, ell in enumerate(lengths):
        x0, x1 = cursor, cursor + ell
        axnet.plot([x0,x1,x1,x0,x0], [0,0,H,H,0], color="black", lw=1.4)
        if show_labels:
            axnet.text((x0+x1)/2, H/2, f"Б{i+1}", ha="center", va="center", fontsize=9)
            axnet.text((x0+x1)/2, -0.12, f"{ell:.2f}", ha="center", va="top", fontsize=8)
        cursor = x1
    axnet.plot([0, perimeter, perimeter, 0, 0], [0,0,H,H,0], color="black", lw=1.8)

    # Основи: жорстко повертаємо контур так, щоб перше ребро збігалося з [0, l1].
    base_net = rotate_to_first_edge(points)
    base_net[:, 1] *= -1
    base_net_closed = np.vstack([base_net, base_net[0]])
    axnet.plot(base_net_closed[:,0], base_net_closed[:,1], color="darkorange", lw=1.8)
    axnet.plot([0, lengths[0]], [0, 0], color="darkorange", lw=2.8)

    # Друга основа приєднується до верхнього ребра першої грані.
    # Вона має ту саму форму, віддзеркалену відносно ребра прикріплення.
    top_net = rotate_to_first_edge(points)
    top_net[:, 1] *= -1
    top_net[:, 1] *= -1
    top_net[:, 1] += H
    top_closed = np.vstack([top_net, top_net[0]])
    axnet.plot(top_closed[:,0], top_closed[:,1], color="darkorange", lw=1.8)
    axnet.plot([0, lengths[0]], [H, H], color="darkorange", lw=2.8)

    if show_labels:
        axnet.text(lengths[0]/2, -max(H, perimeter)*.07, "Основа 1",
                   ha="center", va="top", fontsize=9)
        axnet.text(lengths[0]/2, H+max(H, perimeter)*.07, "Основа 2",
                   ha="center", va="bottom", fontsize=9)
        axnet.text(perimeter/2, H/2, f"Sбіч = P·H = {perimeter*H:.2f}",
                   ha="center", va="center", fontsize=9, alpha=.7)

    margin = max(H, np.max(lengths), 1)*.5
    axnet.set_xlim(-margin, perimeter+margin)
    axnet.set_ylim(-margin, H+margin)
    axnet.set_aspect("equal", adjustable="box")
    axnet.grid(True, alpha=.2)
    axnet.set_xlabel("Координата в площині розгортки")
    axnet.set_ylabel("Координата в площині розгортки")
    axnet.set_title("Розгортка поверхні")
    fig.suptitle("Пряма призма з різнобічною основою", fontsize=14)
    plt.tight_layout()
    plt.show()

    print("Довжини сторін основи:", ", ".join(f"{v:.3f}" for v in lengths))
    print(f"Периметр основи P = {perimeter:.3f}")
    print(f"Висота призми H = {H:.3f}")
    print(f"Площа бічної поверхні Sбіч = P·H = {perimeter:.3f}·{H:.3f} = {perimeter*H:.3f}")

coords_box = Textarea(
    value="-2.2,-1.2; 1.2,-1.6; 2.4,0.4; 0.7,2.0; -1.8,1.4",
    description="Вершини",
    layout={"width": "650px", "height": "90px"}
)
height_box = FloatSlider(min=1, max=8, step=0.5, value=4, description="Висота H")
labels_box = Checkbox(value=True, description="Підписи")
run_button = Button(description="Побудувати", button_style="primary")
out = Output()

def on_build(_):
    with out:
        out.clear_output(wait=True)
        try:
            pts = []
            for item in coords_box.value.split(";"):
                x, y = item.strip().split(",")
                pts.append([float(x), float(y)])
            build_prism_net(np.array(pts), height_box.value, labels_box.value)
        except Exception as err:
            print("Помилка введення:", err)

run_button.on_click(on_build)
display(VBox([coords_box, height_box, labels_box, run_button, out]))


## Практичні завдання

1. Побудуйте розгортку для початкової п'ятикутної основи.
2. Змініть координати однієї вершини, наприклад `2.4,0.4`. Перевірте, як змінюються довжини сторін.
3. Задайте три вершини, щоб отримати трикутну призму; потім чотири — щоб отримати чотирикутну.
4. Змініть висоту `H`. Чи змінюються ширини прямокутників бічної поверхні?
5. Перевірте обчислення площі бічної поверхні за формулою `Sбіч = P·H`.

**Формат координат:** `x1,y1; x2,y2; x3,y3; ...`. Вершини потрібно вводити послідовно по контуру, без перетину ребер. Для цього тренажера основа має бути опуклою.

**Зверніть увагу:** це пряма призма з різнобічною основою. Розгортка бічної поверхні складається з прямокутників різної ширини, але однакової висоти.
